In [2]:
# Kaggle usually has cupy pre-installed, but this ensures it matches the active CUDA driver
!pip install -q numba
!pip install -q cupy-cuda12x || pip install -q cupy-cuda11x

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import numpy as np
import cupy as cp
from numba import cuda
import time

# Using Keras for the fastest possible dataset download
from tensorflow.keras.datasets import mnist

2026-06-01 06:16:33.318942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780294593.821794      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780294593.965265      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780294595.272933      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780294595.272975      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780294595.272978      58 computation_placer.cc:177] computation placer alr

In [20]:
print("Importing the MNIST dataset...")

# Load dataset (downloads a lightweight .npz file)
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Cast the data to float32 for GPU processing
images = train_images.astype(np.float32)

# Keras natively loads them as 2D spatial images (60000, 28, 28)
THRESHOLD = 127.0

# We will process the entire training set of 60,000 images
batch_images = images 
print(f"Dataset successfully imported! Processing shape: {batch_images.shape}")

Importing the MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset successfully imported! Processing shape: (60000, 28, 28)


In [22]:
def count_with_cupy(images_np, threshold):
    # Move array to Kaggle's GPU
    d_images = cp.asarray(images_np)
    
    start_time = time.time()
    
    # Vectorized computation on GPU
    counts = cp.sum(d_images > threshold, axis=(1, 2))
    
    # Synchronize to ensure accurate timing measurement
    cp.cuda.Stream.null.synchronize() 
    end_time = time.time()
    
    # Bring results back to CPU
    return counts.get(), end_time - start_time

In [23]:
@cuda.jit
def count_above_threshold_kernel(images, threshold, counts):
    # Extract 3D grid coordinates: image_index (z), row (y), col (x)
    img_idx, row, col = cuda.grid(3)
    
    # Prevent out-of-bounds memory access
    if (img_idx < images.shape[0] and 
        row < images.shape[1] and 
        col < images.shape[2]):
        
        # Check pixel intensity
        if images[img_idx, row, col] > threshold:
            # Safely increment the count for this specific image using atomic add
            cuda.atomic.add(counts, img_idx, 1)

def count_with_numba(images_np, threshold):
    N, rows, cols = images_np.shape
    
    # Transfer data to GPU and allocate results array
    d_images = cuda.to_device(images_np)
    d_counts = cuda.to_device(np.zeros(N, dtype=np.int32))
    
    # Define thread hierarchy (8x8x8 = 512 threads per block)
    threads_per_block = (8, 8, 8)
    
    # Calculate grid dimensions
    blocks_per_grid_z = int(np.ceil(N / threads_per_block[0]))
    blocks_per_grid_y = int(np.ceil(rows / threads_per_block[1]))
    blocks_per_grid_x = int(np.ceil(cols / threads_per_block[2]))
    blocks_per_grid = (blocks_per_grid_z, blocks_per_grid_y, blocks_per_grid_x)
    
    start_time = time.time()
    
    # Execute Kernel
    count_above_threshold_kernel[blocks_per_grid, threads_per_block](d_images, threshold, d_counts)
    cuda.synchronize()
    
    end_time = time.time()
    
    # Bring results back to CPU
    return d_counts.copy_to_host(), end_time - start_time

In [24]:
print("Warming up GPU context...")
# Warm-up calls to compile the kernels and initialize memory contexts
_, _ = count_with_cupy(batch_images[:10], THRESHOLD)
_, _ = count_with_numba(batch_images[:10], THRESHOLD)

print("\nRunning benchmarks on 60,000 images...")

# Run CuPy
cupy_counts, cupy_time = count_with_cupy(batch_images, THRESHOLD)
print(f"CuPy execution time:  {cupy_time:.6f} seconds")

# Run Numba
numba_counts, numba_time = count_with_numba(batch_images, THRESHOLD)
print(f"Numba execution time: {numba_time:.6f} seconds")

# Verify accuracy
match = np.array_equal(cupy_counts, numba_counts)
print(f"\nResults match perfectly: {match}")
print(f"Sample pixel counts (first 5 images): {cupy_counts[:5]}")

Warming up GPU context...


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 32 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))



Running benchmarks on 60,000 images...
CuPy execution time:  0.021307 seconds
Numba execution time: 0.003989 seconds

Results match perfectly: True
Sample pixel counts (first 5 images): [111 125  81  66  91]
